# NeuroFetal AI — Calibration & Ensemble Retraining

**Version 5.0** — Calibration Fix Phase

This notebook retrains the diverse ensemble with calibration and uncertainty fixes:

### What Changed
| # | Fix | File | Effect |
|---|-----|------|--------|
| 1 | Isotonic Regression on meta-learner | `train_diverse_ensemble.py` | Well-calibrated probabilities |
| 2 | Ensemble variance uncertainty | `evaluate_uncertainty.py` | Uncertainty = std(ResNet, Inception, XGB) |
| 3 | Focal Loss γ reduced (2.0 → 1.5) | `focal_loss.py` | Less overconfident predictions |
| 4 | Ensemble inference in app | `app.py` + `model_loader.py` | Live uncertainty from 3 models |

### Pipeline Steps
| # | Phase | Script |
|---|-------|--------|
| 1 | Setup | Clone repo, install deps |
| 2 | Data Ingestion | `data_ingestion.py` |
| 3 | SSL Pretraining | `pretrain.py` |
| 4 | Primary Training (γ=1.5) | `train.py` |
| 5 | Ensemble Training + Isotonic Calibration | `train_diverse_ensemble.py` |
| 6 | Uncertainty Evaluation | `evaluate_uncertainty.py` |
| 7 | Push Models to GitHub | `git push` |

## 1. Setup Environment

In [ ]:
from google.colab import userdata
import os

# 1. GitHub Authentication
GITHUB_REPO = "Krishna200608/NeuroFetal-AI"

try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    print("✓ GitHub Token loaded from Secrets.")
except Exception as e:
    print("⚠️ Error loading GITHUB_TOKEN from Secrets. Falling back to manual input.")
    from getpass import getpass
    GITHUB_TOKEN = getpass("Enter GitHub Personal Access Token (PAT): ")

os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
os.environ['GITHUB_REPO'] = GITHUB_REPO

In [ ]:
# 2. Clone Repository & Checkout main branch
import shutil
import os

# Reset to /content before deleting the repo folder
try:
    os.chdir("/content")
except:
    pass

# Clean up any previous clone
if os.path.exists("/content/NeuroFetal-AI"):
    shutil.rmtree("/content/NeuroFetal-AI")

print("Cloning repository...")
!git clone https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git

os.chdir("/content/NeuroFetal-AI")

# Checkout main branch
!git checkout main
!git pull origin main
print("✓ Cloned and checked out main!")

### 1.5 Git Credentials

In [ ]:
!git config --global user.email "krishnasikheriya001@gmail.com"
!git config --global user.name "Krishna200608"
print("✓ Git credentials set.")

### 1.6 Install Dependencies

In [ ]:
print("Installing libraries...")
!pip install -q wfdb shap scipy imbalanced-learn pyngrok filterpy \
    scikit-learn matplotlib seaborn pandas numpy tensorflow \
    streamlit plotly python-dotenv xgboost lightgbm
print("✓ Dependencies installed.")

---
## 2. Data Ingestion

Processes raw `.dat`/`.hea` files into clean `.npy` arrays (18 features, pH 7.15, quality filter).

In [ ]:
!python Code/scripts/data_ingestion.py

---
## 3. Self-Supervised Pretraining

Train the Masked Autoencoder (MAE) on FHR data.
Saves encoder weights → `Code/models/pretrained_fhr_encoder.weights.keras`

In [ ]:
!python Code/scripts/pretrain.py

---
## 4. Primary Model Training (Focal Loss γ=1.5)

Train **AttentionFusionResNet** with 5-Fold CV + TimeGAN augmentation.

**V5.0 change:** Focal Loss γ reduced from 2.0 → 1.5 to reduce overconfidence.

In [ ]:
# Pull latest changes (includes calibration fixes)
!git pull origin main

In [ ]:
!python Code/scripts/train.py

---
## 5. Diverse Ensemble Training + Isotonic Calibration

Trains 3 diverse models and stacks with **CalibratedClassifierCV (Isotonic Regression)**:
- Model A: AttentionFusionResNet (from Step 4)
- Model B: 1D-InceptionNet
- Model C: XGBoost

The meta-learner is now wrapped with `CalibratedClassifierCV(method='isotonic', cv=5)` for
perfectly calibrated output probabilities.

**Outputs:**
- `Code/models/inception_model_fold_*.keras`
- `Code/models/xgboost_model_fold_*.pkl`
- `Code/models/stacking_meta_learner.pkl` (calibrated)

In [ ]:
!python Code/scripts/train_diverse_ensemble.py

---
## 6. Uncertainty Evaluation (Ensemble Variance)

Evaluates uncertainty using **ensemble disagreement** (std-dev across 3 base models)
instead of the old MC Dropout approach.

**Outputs:**
- Calibration curves per fold
- Uncertainty histograms per fold
- Expected Calibration Error (ECE)

In [ ]:
!python Code/scripts/evaluate_uncertainty.py

---
## 7. Push Trained Models to GitHub

Commit all trained models and evaluation reports back to the repository.

In [ ]:
!git add Code/models/*.keras Code/models/*.pkl Code/models/*.npy
!git add Reports/uncertainty_analysis/ -f
!git commit -m "V5.0: Retrained ensemble with Isotonic calibration + γ=1.5"
!git push origin main
print("✓ Models and reports pushed to GitHub!")

---
## ✅ Done!

After this notebook completes:
1. All 3 ensemble models are retrained with reduced Focal Loss γ
2. The stacking meta-learner is calibrated with Isotonic Regression
3. Uncertainty is computed via ensemble variance (not MC Dropout)
4. Models are pushed to GitHub — pull locally and run the Streamlit app to verify